# 03. Merged Strict DET + Cloud Auxiliary Feedback GA Search Tutorial

This notebook replaces `tutorial/03_merged_feedback_ga_search.ipynb`.

Strict DET is primary. Cloud/advisor feedback is auxiliary.

In [ ]:
# ============================================================
# Canonical JOILang GA Search Tutorial Setup
# Runtime: JOILang-Server/utils and JOILang-Server/utils/ga_search only
# ============================================================

import os
import sys
import json
import shlex
import time
import html
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 260)

# ------------------------------------------------------------
# Server preset
# ------------------------------------------------------------
SERVER_PRESET = os.environ.get("SERVER_PRESET", "a100")  # "a100" or "a6000"

if SERVER_PRESET == "a100":
    os.environ.setdefault("JOILANG_BASE_DIR", "/root/llm/JOILang-Server")
    os.environ.setdefault("JOI_GA_WORKER_PYTHON", "/root/miniconda3/bin/python")
    os.environ.setdefault("JOI_GA_LOCAL_MODEL_NAME", "/root/llm/local_models/qwen25_coder_14b")
elif SERVER_PRESET == "a6000":
    os.environ.setdefault("JOILANG_BASE_DIR", "/home/mgjeong/Desktop/llm/JOILang-Server")
    os.environ.setdefault("JOI_GA_WORKER_PYTHON", "/home/mgjeong/miniconda3/envs/l/bin/python")
    os.environ.setdefault("JOI_GA_LOCAL_MODEL_NAME", "/home/mgjeong/Desktop/llm/local_models/qwen25_coder_14b")
else:
    raise ValueError(f"Unknown SERVER_PRESET={SERVER_PRESET!r}")

BASE_DIR = Path(os.environ["JOILANG_BASE_DIR"]).expanduser().resolve()
GA_CLI = BASE_DIR / "utils" / "ga_search" / "cli.py"
DATASET = BASE_DIR / "datasets" / "JOICommands-280.csv"
SERVICE_SCHEMA = BASE_DIR / "datasets" / "service_list_ver2.0.1.json"

# Model package can be dotted module or path.
# Examples:
#   gpt_mg.version0_13
#   gpt_cap.stage_2
#   gpt_mg/version0_13
MODEL = os.environ.get("JOI_GA_MODEL", "gpt_mg.version0_13")
MODEL_KEY = os.environ.get("MODEL_KEY", "qwen25_coder_14b")

# Keep mock as safe default. Use worker/openai/local only after smoke passes.
LLM_MODE = os.environ.get("JOI_GA_LLM_MODE", "mock")
ENGINE_MODE = os.environ.get("JOI_GA_ENGINE_MODE", "auto")  # auto|mock|real

RUN_TAG = os.environ.get("RUN_TAG", datetime.now().strftime("%Y%m%d_%H%M%S"))
NB_ROOT = BASE_DIR / "artifacts" / "ga_search_tutorial_runs" / RUN_TAG
NB_ROOT.mkdir(parents=True, exist_ok=True)

PY = os.environ.get("PY", sys.executable)

ENV = os.environ.copy()
ENV.update({
    "PYTHONFAULTHANDLER": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "TRANSFORMERS_VERBOSITY": "error",
})

def ts():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def run_cmd(cmd, *, cwd=BASE_DIR, log_path=None, check=True, timeout_sec=None, heartbeat_sec=30):
    cmd = [str(x) for x in cmd]
    print("\n[CMD]")
    print(" ".join(shlex.quote(x) for x in cmd))
    if log_path:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        print("[LOG]", log_path)

    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    start = time.time()
    last_heartbeat = start
    lines = []

    with (open(log_path, "w", encoding="utf-8") if log_path else open(os.devnull, "w", encoding="utf-8")) as lf:
        assert proc.stdout is not None
        while True:
            line = proc.stdout.readline()
            if line:
                print(line, end="")
                lines.append(line)
                if log_path:
                    lf.write(line)
                    lf.flush()
            elif proc.poll() is not None:
                break
            else:
                now = time.time()
                if timeout_sec and now - start > timeout_sec:
                    proc.kill()
                    raise TimeoutError(f"command timeout after {timeout_sec}s")
                if now - last_heartbeat >= heartbeat_sec:
                    print(f"[still running] elapsed={int(now-start)}s pid={proc.pid}")
                    last_heartbeat = now
                time.sleep(0.25)

    rc = proc.wait()
    out = "".join(lines)
    if check and rc != 0:
        raise RuntimeError(f"command failed rc={rc}: {' '.join(cmd)}")
    return rc, out

def out_dir(name):
    p = NB_ROOT / name
    p.mkdir(parents=True, exist_ok=True)
    return p

def cli_base(command):
    return [PY, "-m", "utils.ga_search.cli", command]

def load_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

def read_csv(path):
    path = Path(path)
    return pd.read_csv(path) if path.exists() and path.stat().st_size > 0 else pd.DataFrame()

def show_file(path, max_chars=5000):
    path = Path(path)
    print(path, "exists=", path.exists(), "size=", path.stat().st_size if path.exists() else 0)
    if path.exists():
        text = path.read_text(encoding="utf-8", errors="replace")
        print(text[:max_chars])
        if len(text) > max_chars:
            print(f"\n... truncated {len(text)-max_chars} chars")

def run_render(label="render_smoke", user_input="Turn on the light.", dry_run=True):
    od = out_dir(label)
    cmd = cli_base("render") + [
        "--model", MODEL,
        "--user-input", user_input,
        "--search-mode", "auto",
    ]
    if dry_run:
        cmd.append("--dry-run")
    rc, out = run_cmd(cmd, log_path=od / f"{label}.log", check=True)
    return od

def run_eval(
    label,
    *,
    row_no=None,
    category=None,
    limit_per_category=None,
    sample_size=None,
    llm_mode=None,
    engine_mode=None,
    llm_extra_json=None,
    llm_endpoint=None,
    print_mode="summary",
    check=True,
    timeout_sec=None,
):
    od = out_dir(label)
    cmd = cli_base("eval") + [
        "--model", MODEL,
        "--dataset", str(DATASET),
        "--service-schema", str(SERVICE_SCHEMA),
        "--llm-mode", llm_mode or LLM_MODE,
        "--engine-mode", engine_mode or ENGINE_MODE,
        "--model-key", MODEL_KEY,
        "--det-profile", "strict",
        "--det-threshold", "70",
        "--out-dir", str(od),
        "--print-mode", print_mode,
    ]
    if row_no is not None:
        cmd += ["--row-no", str(row_no)]
    if category is not None:
        cmd += ["--category", str(category)]
    if limit_per_category is not None:
        cmd += ["--limit-per-category", str(limit_per_category)]
    if sample_size is not None:
        cmd += ["--sample-size", str(sample_size)]
    if llm_extra_json:
        cmd += ["--llm-extra-json", str(llm_extra_json)]
    if llm_endpoint:
        cmd += ["--llm-endpoint", str(llm_endpoint)]
    rc, out = run_cmd(cmd, log_path=od / f"{label}.log", check=check, timeout_sec=timeout_sec)
    return od, rc

def run_search(
    label,
    *,
    row_no=None,
    category=None,
    limit_per_category=None,
    population=2,
    gens=1,
    llm_mode=None,
    engine_mode=None,
    advisor_mode="none",
    advisor_llm_mode="mock",
    prompt_patches=None,
    llm_extra_json=None,
    llm_endpoint=None,
    print_mode="summary",
    check=True,
    timeout_sec=None,
):
    od = out_dir(label)
    cmd = cli_base("search") + [
        "--model", MODEL,
        "--dataset", str(DATASET),
        "--service-schema", str(SERVICE_SCHEMA),
        "--search-mode", "auto",
        "--llm-mode", llm_mode or LLM_MODE,
        "--engine-mode", engine_mode or ENGINE_MODE,
        "--model-key", MODEL_KEY,
        "--det-profile", "strict",
        "--det-threshold", "70",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--population", str(population),
        "--gens", str(gens),
        "--advisor-mode", advisor_mode,
        "--advisor-llm-mode", advisor_llm_mode,
        "--out-dir", str(od),
        "--print-mode", print_mode,
    ]
    if row_no is not None:
        cmd += ["--row-no", str(row_no)]
    if category is not None:
        cmd += ["--category", str(category)]
    if limit_per_category is not None:
        cmd += ["--limit-per-category", str(limit_per_category)]
    if prompt_patches:
        cmd += ["--prompt-patches", str(prompt_patches)]
    if llm_extra_json:
        cmd += ["--llm-extra-json", str(llm_extra_json)]
    if llm_endpoint:
        cmd += ["--llm-endpoint", str(llm_endpoint)]
    rc, out = run_cmd(cmd, log_path=od / f"{label}.log", check=check, timeout_sec=timeout_sec)
    return od, rc

def run_advisor(label, *, advisor_mode="hybrid", strict_results_dir=None, cloud_judge_csv=None, advisor_rich_feedback=None, prompt_patches=None, run_dir=None):
    od = out_dir(label)
    cmd = cli_base("advisor") + [
        "--model", MODEL,
        "--advisor-mode", advisor_mode,
        "--advisor-llm-mode", "mock",
        "--dry-run-advisor",
        "--out-dir", str(od),
    ]
    if strict_results_dir:
        cmd += ["--strict-results-dir", str(strict_results_dir)]
    if cloud_judge_csv:
        cmd += ["--cloud-judge-csv", str(cloud_judge_csv)]
    if advisor_rich_feedback:
        cmd += ["--advisor-rich-feedback", str(advisor_rich_feedback)]
    if prompt_patches:
        cmd += ["--prompt-patches", str(prompt_patches)]
    if run_dir:
        cmd += ["--run-dir", str(run_dir)]
    rc, out = run_cmd(cmd, log_path=od / f"{label}.log", check=True)
    return od

def run_check(label, check_name, *, advisor_dir=None, run_dir=None, check=True):
    od = out_dir(label)
    cmd = cli_base("check") + ["--model", MODEL, "--check", check_name, "--out-dir", str(od)]
    if advisor_dir:
        cmd += ["--advisor-dir", str(advisor_dir)]
    if run_dir:
        cmd += ["--run-dir", str(run_dir)]
    rc, out = run_cmd(cmd, log_path=od / f"{label}.log", check=check)
    return od, rc

def run_patch_apply(label, patches_path, genome_json=None):
    od = out_dir(label)
    cmd = cli_base("patch-apply") + [
        "--model", MODEL,
        "--prompt-patches", str(patches_path),
        "--out-dir", str(od),
    ]
    if genome_json:
        cmd += ["--genome-json", str(genome_json)]
    rc, out = run_cmd(cmd, log_path=od / f"{label}.log", check=True)
    return od

def candidates_df(run_dir):
    return read_csv(Path(run_dir) / "candidates" / "generation_000.csv")

def eval_df(run_dir):
    return read_csv(Path(run_dir) / "eval" / "row_evaluation.csv")

def summary_json(run_dir):
    for p in [
        Path(run_dir) / "ga_summary.json",
        Path(run_dir) / "eval" / "summary.json",
        Path(run_dir) / "manifest.json",
    ]:
        if p.exists():
            return load_json(p)
    return {}

def gt_pretty(raw):
    if raw is None:
        return ""
    if not isinstance(raw, str):
        raw = json.dumps(raw, ensure_ascii=False)
    raw = raw.strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict):
            script = obj.get("script", obj.get("code", ""))
            meta = dict(obj)
            meta.pop("script", None)
            meta.pop("code", None)
            return "[JSON meta]\n" + json.dumps(meta, ensure_ascii=False, indent=2) + "\n\n[script/code]\n" + str(script).replace("\\n", "\n")
        return json.dumps(obj, ensure_ascii=False, indent=2)
    except Exception:
        return raw.replace("\\n", "\n").replace("\\t", "    ")

def _pre(title, text):
    return f"""
    <div>
      <div style="font-weight:700;background:#f7f7f7;padding:6px;border:1px solid #ddd;border-bottom:none;">{html.escape(str(title))}</div>
      <pre style="margin:0;white-space:pre;overflow:auto;max-height:520px;min-height:160px;tab-size:4;font-family:Consolas,'Courier New',monospace;font-size:13px;line-height:1.45;background:#fbfbfb;padding:10px;border:1px solid #ddd;">{html.escape(gt_pretty(text))}</pre>
    </div>
    """

def show_gt_vs_generated(run_dir, row_no=None, max_rows=20):
    cdf = candidates_df(run_dir)
    edf = eval_df(run_dir)
    if cdf.empty:
        print("No candidates found:", run_dir)
        return pd.DataFrame()
    if row_no is not None and "row_no" in cdf.columns:
        cdf = cdf[cdf["row_no"].astype(str) == str(row_no)]
    if not edf.empty:
        join_cols = [c for c in ["row_no", "genome_id", "candidate_index"] if c in cdf.columns and c in edf.columns]
        merged = cdf.merge(edf, on=join_cols, how="left", suffixes=("", "_eval")) if join_cols else cdf
    else:
        merged = cdf
    cards = []
    for _, r in merged.head(max_rows).iterrows():
        gt = r.get("gt", "")
        generated = r.get("generated_json", "") or r.get("candidates", "") or r.get("generated_code", "")
        title = f"row={r.get('row_no')} cat={r.get('category')} genome={r.get('genome_id')} det={r.get('det_score', '')} pass={r.get('det_pass', '')}"
        cards.append(f"""
        <div style="border:1px solid #ccc;border-radius:8px;padding:12px;margin:12px 0;">
          <div style="font-weight:700;margin-bottom:8px;">{html.escape(str(title))}</div>
          <div style="display:grid;grid-template-columns:minmax(0,1fr) minmax(0,1fr);gap:12px;">
            {_pre("GT", gt)}
            {_pre("Generated", generated)}
          </div>
          <div style="margin-top:8px;font-size:13px;"><b>failure_reasons:</b> {html.escape(str(r.get('failure_reasons', '')))}</div>
          <div style="margin-top:4px;font-size:13px;"><b>prompt_log_paths:</b> {html.escape(str(r.get('prompt_log_paths', '')))}</div>
          <div style="margin-top:4px;font-size:13px;"><b>raw_response_path:</b> {html.escape(str(r.get('raw_response_path', '')))}</div>
        </div>
        """)
    display(HTML("\n".join(cards)))
    return merged

def row_summary(run_dir, max_rows=120):
    edf = eval_df(run_dir)
    if edf.empty:
        print("No eval rows")
        return edf
    cols = [c for c in [
        "row_no","category","genome_id","candidate_index","det_score","det_pass","gt_exact",
        "gt_similarity","schedule_match","service_recall","service_precision","receiver_recall",
        "numeric_grounding","enum_grounding","dataflow_score","failure_reasons","generated_code","gt_code"
    ] if c in edf.columns]
    display(edf[cols].head(max_rows))
    return edf

def failed_or_low_rows(run_dir, threshold=70.0):
    edf = eval_df(run_dir)
    if edf.empty or "row_no" not in edf.columns:
        return []
    d = edf.copy()
    d["det_score_num"] = pd.to_numeric(d.get("det_score", 0), errors="coerce").fillna(0)
    det_pass_text = d.get("det_pass", pd.Series(["false"] * len(d))).astype(str).str.lower()
    failed = d[(det_pass_text != "true") | (d["det_score_num"] < threshold)]
    if failed.empty:
        failed = d.sort_values("det_score_num", ascending=True)
    return failed["row_no"].dropna().astype(int).drop_duplicates().tolist()

def compare_eval_runs(before_dir, after_dir):
    b = eval_df(before_dir)
    a = eval_df(after_dir)
    if b.empty or a.empty:
        print("Missing eval outputs")
        return pd.DataFrame()
    keys = [k for k in ["row_no","genome_id","candidate_index"] if k in b.columns and k in a.columns]
    if not keys:
        keys = ["row_no"]
    merged = b.merge(a, on=keys, how="outer", suffixes=("_before","_after"))
    display(merged.head(120))
    return merged

def show_artifact_table(paths):
    display(pd.DataFrame([{"path": str(p), "exists": Path(p).exists(), "size": Path(p).stat().st_size if Path(p).exists() else 0} for p in paths]))

def worker_extra_json_path(label="worker_extra"):
    p = out_dir(label) / "llm_extra_worker.json"
    payload = {
        "local_worker": os.environ.get("JOI_GA_WORKER_PATH", "gpt_mg/version0_13/qwen_local_worker.py"),
        "worker_python": os.environ.get("JOI_GA_WORKER_PYTHON", sys.executable),
        "local_model_name": os.environ.get("JOI_GA_LOCAL_MODEL_NAME", ""),
        "local_device": os.environ.get("JOI_GA_LOCAL_DEVICE", "cuda:0"),
        "local_files_only": True,
        "local_trust_remote_code": True,
    }
    p.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("worker extra json:", p)
    print(json.dumps(payload, ensure_ascii=False, indent=2))
    return p

print("BASE_DIR:", BASE_DIR)
print("GA_CLI:", GA_CLI, GA_CLI.exists())
print("DATASET:", DATASET, DATASET.exists())
print("SERVICE_SCHEMA:", SERVICE_SCHEMA, SERVICE_SCHEMA.exists())
print("MODEL:", MODEL)
print("MODEL_KEY:", MODEL_KEY)
print("LLM_MODE:", LLM_MODE)
print("ENGINE_MODE:", ENGINE_MODE)
print("NB_ROOT:", NB_ROOT)

assert BASE_DIR.exists(), BASE_DIR
assert GA_CLI.exists(), GA_CLI
assert DATASET.exists(), DATASET

## 1. Baseline strict DET evaluation

In [ ]:
CATEGORY_TO_RUN = 6
LIMIT_PER_CATEGORY = 5

baseline_eval_dir, _ = run_eval(
    label=f"merged_baseline_eval_cat{CATEGORY_TO_RUN}_{ts()}",
    category=CATEGORY_TO_RUN,
    limit_per_category=LIMIT_PER_CATEGORY,
    llm_mode="mock",
    engine_mode="mock",
    print_mode="summary",
)
baseline_eval = row_summary(baseline_eval_dir)
INSPECT_ROWS = failed_or_low_rows(baseline_eval_dir, threshold=70.0)[:5]
print("INSPECT_ROWS:", INSPECT_ROWS)
for r in INSPECT_ROWS[:3]:
    _ = show_gt_vs_generated(baseline_eval_dir, row_no=r, max_rows=6)

## 2. Hybrid advisor evidence and patches

In [ ]:
CLOUD_JUDGE_CSV = os.environ.get("CLOUD_JUDGE_CSV", "")
ADVISOR_RICH_FEEDBACK = os.environ.get("ADVISOR_RICH_FEEDBACK", "")

hybrid_advisor_dir = run_advisor(
    label=f"merged_hybrid_advisor_{ts()}",
    advisor_mode="hybrid",
    strict_results_dir=str(baseline_eval_dir),
    cloud_judge_csv=CLOUD_JUDGE_CSV or None,
    advisor_rich_feedback=ADVISOR_RICH_FEEDBACK or None,
)
show_file(hybrid_advisor_dir / "advisor" / "advisor_evidence_packet.json", max_chars=4000)
show_file(hybrid_advisor_dir / "advisor" / "prompt_patches.json", max_chars=4000)

## 3. Baseline search vs hybrid-patched search

In [ ]:
baseline_search_dir, _ = run_search(
    label=f"merged_baseline_search_cat{CATEGORY_TO_RUN}_{ts()}",
    category=CATEGORY_TO_RUN,
    limit_per_category=LIMIT_PER_CATEGORY,
    population=2,
    gens=1,
    llm_mode="mock",
    engine_mode="mock",
    advisor_mode="none",
    print_mode="summary",
)

hybrid_patches = hybrid_advisor_dir / "advisor" / "prompt_patches.json"
hybrid_search_dir, _ = run_search(
    label=f"merged_hybrid_search_cat{CATEGORY_TO_RUN}_{ts()}",
    category=CATEGORY_TO_RUN,
    limit_per_category=LIMIT_PER_CATEGORY,
    population=2,
    gens=1,
    llm_mode="mock",
    engine_mode="mock",
    advisor_mode="hybrid",
    prompt_patches=hybrid_patches,
    print_mode="summary",
)

print("baseline ga_summary:")
print(json.dumps(load_json(baseline_search_dir / "ga_summary.json"), ensure_ascii=False, indent=2))
print("hybrid ga_summary:")
print(json.dumps(load_json(hybrid_search_dir / "ga_summary.json"), ensure_ascii=False, indent=2))

_ = compare_eval_runs(baseline_search_dir, hybrid_search_dir)

## 4. Patch visibility and advisor effectiveness

In [ ]:
show_artifact_table([
    hybrid_search_dir / "patch_application_report.json",
    hybrid_search_dir / "patch_diff.md",
    hybrid_search_dir / "patched_prompt_preview.md",
    hybrid_search_dir / "advisor" / "mutation_population.json",
    hybrid_search_dir / "ga_summary.json",
])

show_file(hybrid_search_dir / "patch_application_report.json", max_chars=5000)

transport_dir, transport_rc = run_check(
    label=f"merged_transport_check_{ts()}",
    check_name="advisor_transport_smoke",
    run_dir=hybrid_search_dir,
    check=False,
)
effect_dir, effect_rc = run_check(
    label=f"merged_effectiveness_check_{ts()}",
    check_name="advisor_effectiveness_smoke",
    run_dir=hybrid_search_dir,
    check=False,
)
print("transport_rc:", transport_rc)
print("effectiveness_rc:", effect_rc)
show_file(transport_dir / "advisor_transport_smoke.json")
show_file(effect_dir / "advisor_effectiveness_smoke.json")

## 5. Optional real worker comparison

In [ ]:
RUN_WORKER_REAL = False

if RUN_WORKER_REAL:
    extra = worker_extra_json_path("merged_worker_extra")
    worker_dir, worker_rc = run_search(
        label=f"merged_worker_real_cat{CATEGORY_TO_RUN}_{ts()}",
        category=CATEGORY_TO_RUN,
        limit_per_category=1,
        population=2,
        gens=1,
        llm_mode="worker",
        engine_mode="real",
        llm_extra_json=extra,
        advisor_mode="hybrid",
        prompt_patches=hybrid_patches,
        print_mode="summary",
        check=False,
        timeout_sec=1800,
    )
    print("worker_rc:", worker_rc)
    print(json.dumps(load_json(worker_dir / "ga_summary.json"), ensure_ascii=False, indent=2))
    row_summary(worker_dir)
    _ = show_gt_vs_generated(worker_dir, max_rows=10)

## 6. Final artifact checklist

In [ ]:
paths = [
    baseline_eval_dir / "eval" / "summary.json",
    hybrid_advisor_dir / "advisor" / "prompt_patches.json",
    baseline_search_dir / "ga_summary.json",
    hybrid_search_dir / "ga_summary.json",
    hybrid_search_dir / "patch_application_report.json",
    hybrid_search_dir / "advisor" / "mutation_population.csv",
    hybrid_search_dir / "ga_run_manifest.json",
]
show_artifact_table(paths)